# Introduction and Assignment Objective

This notebook completes **Assignment 5: Clustering and Customer Segmentation** for EventEdge Corporate Training.

The purpose of the assignment is to use **unsupervised learning** to discover hidden participant groups in a training-related dataset. The notebook uses **K-means clustering** to identify participant segments based on early engagement, course characteristics, fee information, reminder response, and first-week satisfaction.

The assignment focuses on both the technical clustering workflow and the business interpretation of the results. Therefore, the notebook follows a clear workflow: understanding the business problem, preparing and inspecting the data, selecting useful numerical features, scaling the data, applying the Elbow Method, training a K-means model, interpreting the clusters, visualizing the segments, and explaining how the findings can support EventEdge's marketing and participant-support decisions.

Important note: The dataset includes `High_Dropout_Risk`, but this column is **not used as a target variable** for K-means clustering. It is used only as a reference column for business interpretation after the clusters are created.

# Step 1: Business Problem and Dataset Overview

EventEdge Corporate Training is a B2B learning company that provides professional development programs to client organizations. Its courses include areas such as Data Analytics, Leadership, Project Management, Customer Experience, Digital Marketing, and AI for Business.

The main business problem is **participant dropout and weak course engagement**. When participants disengage or fail to complete a paid training course, EventEdge may face lower client satisfaction, weaker learning outcomes, wasted coaching capacity, and reduced renewal opportunities from client companies.

The business question for this unsupervised learning assignment is:

**Can EventEdge discover meaningful groups of participants based on course information and early engagement behaviour so that the business can design better support and marketing strategies for each group?**

K-means clustering can help EventEdge identify participant segments such as highly engaged learners, premium long-course participants, low-engagement participants, and participants who may need additional reminders or support. These groups can support decisions about coaching, reminders, onboarding, course design, client communication, and resource prioritization.

# Step 2: Import Required Libraries

In this step, the required Python libraries are imported.

The notebook uses `pandas` and `numpy` for data handling, `matplotlib` for charts, `pathlib` for managing output folders, and `scikit-learn` for scaling, K-means clustering, and PCA visualization.

In [ ]:
# Import pandas library to work with datasets
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for charts and visualizations
import matplotlib.pyplot as plt

# Import display function to show tables clearly in Jupyter Notebook
from IPython.display import display

# Import Path to work with file and folder paths
from pathlib import Path

# Import StandardScaler because K-means is distance-based and requires scaled numerical data
from sklearn.preprocessing import StandardScaler

# Import KMeans for clustering
from sklearn.cluster import KMeans

# Import PCA to create a two-dimensional visualization of clusters
from sklearn.decomposition import PCA

# Set pandas option to display all columns when viewing tables
pd.set_option("display.max_columns", None)

# Create the outputs folder if it does not already exist
outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

# Display confirmation message
print("Libraries imported successfully.")
print("Outputs folder is ready.")

# Step 3: Load the Dataset

In this step, the EventEdge training dropout risk dataset is loaded into Python.

The dataset file should be stored in the same folder as this notebook. The data is loaded into a pandas DataFrame named `df`.

Note: If the assigned dataset is provided as an Excel workbook with multiple tabs, use:

`pd.read_excel("eventedge_training_dropout_risk_dataset.xlsx", sheet_name=0)`

This notebook uses a CSV export of the first sheet.

In [ ]:
# Store the dataset file name in a variable
# Important: Keep the dataset file in the same folder as this notebook
file_path = "eventedge_training_dropout_risk_dataset.csv"

# Load the CSV dataset into a pandas DataFrame
# If using Excel with multiple tabs, use: pd.read_excel("eventedge_training_dropout_risk_dataset.xlsx", sheet_name=0)
df = pd.read_csv(file_path)

# Print a message to confirm that the dataset has been loaded successfully
print("Dataset loaded successfully!")

# Print the number of rows and columns in the dataset
print("Dataset shape:", df.shape)

# Display the first five rows of the dataset
display(df.head())

# Step 4: Initial Data Inspection

Before applying K-means clustering, the dataset must be inspected carefully. This step helps us understand the structure of the data, the number of rows and columns, the column names, data types, missing values, duplicate rows, and basic summary statistics.

This inspection is important because K-means clustering requires clean numerical input. Any missing values, duplicate records, irrelevant identifier columns, or unsuitable feature types should be reviewed before selecting features for clustering.

In [ ]:
# Print a title for this output
print("Dataset Shape")

# Print a separator line
print("-" * 50)

# Display the number of rows and columns
print(df.shape)

In [ ]:
# Print a title for this output
print("First Five Rows of the Dataset")

# Print a separator line
print("-" * 50)

# Display the first five rows
display(df.head())

In [ ]:
# Print a title for this output
print("Last Five Rows of the Dataset")

# Print a separator line
print("-" * 50)

# Display the last five rows
display(df.tail())

In [ ]:
# Print a title for this output
print("Column Names")

# Print a separator line
print("-" * 50)

# Display all column names as a list
print(df.columns.tolist())

In [ ]:
# Print a title for this output
print("Dataset Information")

# Print a separator line
print("-" * 50)

# Display column names, non-null counts, and data types
df.info()

In [ ]:
# Print a title for this output
print("Data Types of Each Column")

# Print a separator line
print("-" * 50)

# Display the data type of each column
display(df.dtypes)

In [ ]:
# Print a title for this output
print("Missing Values in Each Column")

# Print a separator line
print("-" * 50)

# Count missing values in each column
missing_values = df.isnull().sum()

# Display missing values
display(missing_values)

In [ ]:
# Print a title for this output
print("Number of Duplicate Rows")

# Print a separator line
print("-" * 50)

# Count duplicate rows
duplicate_rows = df.duplicated().sum()

# Display duplicate row count
print(duplicate_rows)

In [ ]:
# Print a title for this output
print("Summary Statistics for Numerical Columns")

# Print a separator line
print("-" * 50)

# Display summary statistics for numerical columns
display(df.describe())

In [ ]:
# Print a title for this output
print("Summary Statistics for Categorical Columns")

# Print a separator line
print("-" * 50)

# Identify categorical/text columns without using select_dtypes(include=["object"])
# This avoids future pandas string-dtype warnings.
categorical_columns_initial = [
    column for column in df.columns
    if not pd.api.types.is_numeric_dtype(df[column])
]

# Display summary statistics for categorical/text columns
if len(categorical_columns_initial) > 0:
    display(df[categorical_columns_initial].describe())
else:
    print("No categorical columns found.")

## Short Interpretation

The initial inspection shows the size and structure of the EventEdge dataset. The dataset contains both numerical and categorical variables. Since K-means clustering is distance-based, only useful numerical features will be selected for the clustering model.

The inspection also identifies missing values and duplicate rows that need to be reviewed before clustering. The `Participant_ID` column is an identifier and will not be used as a clustering feature. The `High_Dropout_Risk` column is also not used for training because this is an unsupervised learning assignment.

# Step 5: Check Missing Values and Duplicate Rows

In this step, missing values and duplicate rows are reviewed before cleaning the dataset.

Missing values can affect clustering because K-means cannot work properly with empty values. Duplicate rows can also affect clustering because repeated records may give extra weight to the same participant information.

This step creates a clear data-quality review before cleaning is applied.

In [ ]:
# Step 5: Check Missing Values and Duplicate Rows

# Count missing values before cleaning
missing_before_cleaning = df.isnull().sum()

# Select only columns with missing values
missing_before_nonzero = missing_before_cleaning[missing_before_cleaning > 0]

# Count duplicate rows before cleaning
duplicate_rows_before = df.duplicated().sum()

# Print missing values before cleaning
print("Missing Values Before Cleaning")
print("-" * 50)

if len(missing_before_nonzero) > 0:
    display(missing_before_nonzero)
else:
    print("No missing values found in the original dataset.")

# Print duplicate rows before cleaning
print("\nDuplicate Rows Before Cleaning")
print("-" * 50)
print(duplicate_rows_before)

# Create a data-quality review summary table
data_quality_review = pd.DataFrame({
    "Data Quality Item": [
        "Original Dataset Rows",
        "Original Dataset Columns",
        "Columns With Missing Values",
        "Total Missing Values",
        "Duplicate Rows Before Cleaning"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        len(missing_before_nonzero),
        int(missing_before_cleaning.sum()),
        int(duplicate_rows_before)
    ]
})

# Save the data-quality review summary
data_quality_review.to_csv(outputs_dir / "data_quality_review.csv", index=False)

# Display the data-quality review summary
display(data_quality_review)

# Step 6: Clean the Dataset

Before applying K-means clustering, the dataset needs to be cleaned to reduce data-quality issues.

In this step, a copy of the original dataset is created so that the raw data remains unchanged. Then duplicate rows are removed if they exist. Missing values are handled using simple and transparent rules:

- Numerical missing values are filled using the median.
- Categorical missing values are filled using the most frequent value.

For this unsupervised learning assignment, there is no train-test split because the objective is not prediction. Therefore, missing-value handling is completed before clustering so that the selected numerical features can be scaled and used by K-means.

In [ ]:
# Step 6: Clean the Dataset

# Define ID and reference outcome columns
id_column = "Participant_ID"
reference_column = "High_Dropout_Risk"

# Create a copy of the original dataset
# This keeps the original dataset unchanged
df_clean = df.copy()

# Clean column names by removing any extra spaces
df_clean.columns = df_clean.columns.str.strip()

# Check duplicate rows before cleaning
duplicates_before = df_clean.duplicated().sum()

# Remove duplicate rows
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# Check duplicate rows after cleaning
duplicates_after = df_clean.duplicated().sum()

# Check missing values before handling
missing_before = df_clean.isnull().sum()

# Identify numerical columns
numerical_columns_for_cleaning = [
    column for column in df_clean.columns
    if pd.api.types.is_numeric_dtype(df_clean[column])
]

# Identify categorical/text columns
categorical_columns_for_cleaning = [
    column for column in df_clean.columns
    if not pd.api.types.is_numeric_dtype(df_clean[column])
]

# Fill missing values in numerical columns using median
for column in numerical_columns_for_cleaning:
    if df_clean[column].isnull().sum() > 0:
        df_clean[column] = df_clean[column].fillna(df_clean[column].median())

# Fill missing values in categorical/text columns using the most frequent value
for column in categorical_columns_for_cleaning:
    if df_clean[column].isnull().sum() > 0:
        df_clean[column] = df_clean[column].fillna(df_clean[column].mode()[0])

# Check missing values after handling
missing_after = df_clean.isnull().sum()

# Create a duplicate cleaning summary table
duplicate_summary = pd.DataFrame({
    "Cleaning Item": [
        "Duplicate Rows Before Cleaning",
        "Duplicate Rows After Cleaning"
    ],
    "Count": [duplicates_before, duplicates_after]
})

# Create missing value summary table
missing_value_summary = pd.DataFrame({
    "Column": missing_before.index,
    "Missing_Values_Before": missing_before.values,
    "Missing_Values_After": missing_after.reindex(missing_before.index).values
})

# Keep only columns that had missing values before cleaning
missing_value_summary = missing_value_summary[
    missing_value_summary["Missing_Values_Before"] > 0
]

# Create missing-value handling plan table
missing_value_handling_plan = []

for column in missing_value_summary["Column"]:
    if column in numerical_columns_for_cleaning:
        missing_value_handling_plan.append([
            column,
            "Numerical",
            int(missing_before[column]),
            "Median"
        ])
    else:
        missing_value_handling_plan.append([
            column,
            "Categorical",
            int(missing_before[column]),
            "Most frequent value"
        ])

missing_value_handling_plan = pd.DataFrame(
    missing_value_handling_plan,
    columns=["Column", "Data Type", "Missing Values", "Handling Method"]
)

# Save cleaning summaries
missing_value_summary.to_csv(outputs_dir / "missing_value_summary.csv", index=False)
missing_value_handling_plan.to_csv(outputs_dir / "missing_value_handling_plan.csv", index=False)
duplicate_summary.to_csv(outputs_dir / "duplicate_cleaning_summary.csv", index=False)

# Display duplicate cleaning summary
print("Duplicate Cleaning Summary")
print("-" * 50)
display(duplicate_summary)

# Display missing value summary
print("Missing Value Summary")
print("-" * 50)
if missing_value_summary.empty:
    print("No missing values were found before cleaning.")
else:
    display(missing_value_summary)

# Display missing value handling plan
print("Missing Value Handling Plan")
print("-" * 50)
if missing_value_handling_plan.empty:
    print("No missing value handling was required.")
else:
    display(missing_value_handling_plan)

# Display the cleaned dataset shape
print("Cleaned dataset shape:", df_clean.shape)

In [ ]:
# Display the first five rows of the cleaned dataset
display(df_clean.head())

## Short Interpretation

Duplicate rows have been reviewed and removed if they exist. Missing values have also been handled so that the dataset is ready for clustering.

This cleaning step is important because K-means clustering requires complete numerical input. If missing values remain in the selected clustering features, the model will not run correctly.

# Step 7: Chart: Target Variable Distribution

The dataset includes `High_Dropout_Risk`, which shows whether a participant is marked as high dropout risk.

However, this assignment is based on **unsupervised learning**, so `High_Dropout_Risk` is **not used as a target variable for model training**. The chart is included only as a reference to understand the business context after cleaning.

In supervised learning, a target variable is predicted. In this assignment, K-means clustering discovers participant groups without using a target label.

In [ ]:
# Step 7: Chart: Target Variable Distribution

# Check whether the reference outcome column exists
if reference_column in df_clean.columns:

    # Count the number of participants in each High_Dropout_Risk class
    reference_distribution = df_clean[reference_column].value_counts()

    # Print a title
    print("High Dropout Risk Distribution After Cleaning")
    print("-" * 50)

    # Display the distribution
    display(reference_distribution)

    # Create a bar chart for the High_Dropout_Risk distribution
    ax = reference_distribution.plot(
        kind="bar",
        figsize=(6, 4)
    )

    # Add chart title and labels
    plt.title("High Dropout Risk Distribution After Cleaning")
    plt.xlabel("High Dropout Risk")
    plt.ylabel("Number of Participants")

    # Rotate x-axis labels
    plt.xticks(rotation=0)

    # Add number labels on top of bars
    for container in ax.containers:
        ax.bar_label(container)

    # Adjust layout
    plt.tight_layout()

    # Save the chart
    plt.savefig(outputs_dir / "high_dropout_risk_distribution.png", dpi=300, bbox_inches="tight")
    plt.savefig(outputs_dir / "target_distribution_after_cleaning.png", dpi=300, bbox_inches="tight")

    # Display the chart
    plt.show()

else:
    print("The High_Dropout_Risk column is not available in this dataset.")

## Short Interpretation

The chart shows the distribution of `High_Dropout_Risk` after cleaning. This helps describe the business context of participant dropout risk.

However, this column is not used to train the K-means clustering model. The clustering model uses selected numerical features only and discovers groups based on similarity between participant records.

# Step 8: Chart: Missing Values by Column

This step creates a chart showing which columns had missing values before cleaning.

The chart is useful because it visually communicates the main data-quality issues found in the dataset. If there are no missing values, the notebook will print a message instead of creating an empty chart.

In [ ]:
# Step 8: Chart: Missing Values by Column

# Select only columns that had missing values before cleaning
missing_columns = missing_before[missing_before > 0]

# Create a bar chart only if there are columns with missing values
if len(missing_columns) > 0:

    # Create a figure for the missing values chart
    plt.figure(figsize=(8, 4))

    # Create a bar chart for columns with missing values
    ax = missing_columns.plot(kind="bar")

    # Add a title to the chart
    plt.title("Missing Values by Column Before Cleaning")

    # Add a label to the x-axis
    plt.xlabel("Columns")

    # Add a label to the y-axis
    plt.ylabel("Number of Missing Values")

    # Rotate x-axis labels to make them easier to read
    plt.xticks(rotation=45, ha="right")

    # Add value labels on top of bars
    for container in ax.containers:
        ax.bar_label(container)

    # Adjust the layout so labels do not overlap
    plt.tight_layout()

    # Save the chart in the outputs folder
    plt.savefig(outputs_dir / "missing_values_by_column.png", dpi=300, bbox_inches="tight")

    # Display the chart
    plt.show()

else:
    print("No missing values found before cleaning.")

## Short Interpretation

The missing-values chart shows which columns required cleaning before clustering. In this assignment, missing values were handled before feature scaling and K-means clustering.

This step ensures that the selected numerical features are complete and ready for the unsupervised learning model.

# Step 9: Checking Numerical and Categorical Variables

In this step, the cleaned dataset is separated into numerical and categorical columns.

This is important because K-means clustering is a distance-based algorithm and works directly with numerical data. Categorical variables are useful for business interpretation, but they are not included in the final clustering model unless they are encoded carefully.

The `Participant_ID` column is excluded because it is only an identifier, and `High_Dropout_Risk` is excluded because it is a reference outcome column, not an input for clustering.

In [ ]:
# Step 9: Checking Numerical and Categorical Variables

# Create a feature-only dataset by removing the ID and reference outcome columns
feature_data = df_clean.drop(columns=[id_column, reference_column], errors="ignore")

# Select numerical feature columns
numerical_features_initial = [
    column for column in feature_data.columns
    if pd.api.types.is_numeric_dtype(feature_data[column])
]

# Select categorical feature columns
categorical_features_initial = [
    column for column in feature_data.columns
    if not pd.api.types.is_numeric_dtype(feature_data[column])
]

# Print numerical feature columns
print("Numerical Feature Columns")
print("-" * 50)
print(numerical_features_initial)

# Print categorical feature columns
print("\nCategorical Feature Columns")
print("-" * 50)
print(categorical_features_initial)

In [ ]:
# Create a table showing the number of numerical and categorical features
feature_type_summary = pd.DataFrame({
    "Feature Type": ["Numerical", "Categorical"],
    "Number of Features": [len(numerical_features_initial), len(categorical_features_initial)]
})

# Save the feature type summary
feature_type_summary.to_csv(outputs_dir / "feature_type_summary.csv", index=False)

# Display the feature type summary table
display(feature_type_summary)

In [ ]:
# Count unique values for each categorical feature
categorical_unique_counts = feature_data[categorical_features_initial].nunique().sort_values(ascending=False)

# Print a title
print("Number of Unique Values in Each Categorical Feature")

# Print a separator line
print("-" * 50)

# Display the number of unique values
if len(categorical_unique_counts) > 0:
    display(categorical_unique_counts)
else:
    print("No categorical features found.")

In [ ]:
# Create a categorical unique values chart only if categorical features exist
if len(categorical_unique_counts) > 0:

    # Create a figure for the categorical unique values chart
    plt.figure(figsize=(10, 4))

    # Create a bar chart for the number of unique values in each categorical feature
    ax = categorical_unique_counts.plot(kind="bar")

    # Add a title to the chart
    plt.title("Number of Unique Values in Each Categorical Feature")

    # Add a label to the x-axis
    plt.xlabel("Categorical Features")

    # Add a label to the y-axis
    plt.ylabel("Number of Unique Values")

    # Rotate x-axis labels to make them easier to read
    plt.xticks(rotation=45, ha="right")

    # Add value labels on top of bars
    for container in ax.containers:
        ax.bar_label(container)

    # Adjust the layout so labels do not overlap
    plt.tight_layout()

    # Save the chart in the outputs folder
    plt.savefig(outputs_dir / "categorical_unique_values.png", dpi=300, bbox_inches="tight")

    # Display the chart
    plt.show()

else:
    print("No categorical unique values chart was created because no categorical features were found.")

## Short Interpretation

The cleaned dataset contains both numerical and categorical variables. Since K-means clustering is based on distance calculations, the final clustering model will use selected numerical features.

Categorical variables are still useful for business understanding, but they are not used directly in this basic K-means model.

# Step 10: Select Numerical Features for Clustering

In this step, useful numerical features are selected for K-means clustering.

The selected features should describe participant behaviour, course characteristics, and early engagement. The `Participant_ID`, `Enrollment_Date`, categorical variables, and `High_Dropout_Risk` are not used as clustering inputs.

Selected clustering features:

- `Course_Length_Weeks`
- `Fee_Paid`
- `Days_Between_Enrollment_and_Start`
- `Sessions_Attended_First_Week`
- `LMS_Logins_First_Week`
- `Reminder_Emails_Opened`
- `Satisfaction_Score_Week1`

In [ ]:
# Step 10: Select Numerical Features for Clustering

# Create a list of selected numerical features for K-means clustering
clustering_features = [
    "Course_Length_Weeks",
    "Fee_Paid",
    "Days_Between_Enrollment_and_Start",
    "Sessions_Attended_First_Week",
    "LMS_Logins_First_Week",
    "Reminder_Emails_Opened",
    "Satisfaction_Score_Week1"
]

# Create the feature matrix for clustering
X = df_clean[clustering_features].copy()

# Print a confirmation message
print("Selected numerical features for clustering:")
print("-" * 50)
print(clustering_features)

# Display the shape of X
print("\nSelected feature dataset shape:", X.shape)

# Display the first five rows of selected features
display(X.head())

In [ ]:
# Create a summary table for selected clustering features
selected_feature_summary = X.describe().T.round(2)

# Save selected feature summary
selected_feature_summary.to_csv(outputs_dir / "selected_feature_summary.csv")

# Display selected feature summary
display(selected_feature_summary)

## Short Interpretation

The selected numerical features represent course duration, course fee, enrollment timing, first-week attendance, LMS activity, email engagement, and first-week satisfaction.

These variables are appropriate for clustering because they describe differences in participant behaviour and course context. Since K-means uses distance calculations, these features must be scaled before clustering.

# Step 11: Scale the Selected Features

K-means clustering is a distance-based algorithm. This means variables with larger numerical ranges can dominate the clustering result if the data is not scaled.

For example, `Fee_Paid` has much larger values than `Satisfaction_Score_Week1`. Without scaling, the model may give too much importance to course fees and too little importance to engagement or satisfaction.

Therefore, `StandardScaler` is used to place the selected numerical features on a comparable scale.

In [ ]:
# Step 11: Scale the Selected Features

# Create the scaler object
scaler = StandardScaler()

# Fit and transform the selected numerical features
X_scaled = scaler.fit_transform(X)

# Convert the scaled array back into a DataFrame for easier viewing
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)

# Display the first five rows of scaled features
print("Scaled feature data created successfully.")
display(X_scaled_df.head())

## Short Interpretation

The selected numerical features have been scaled. This helps ensure that each feature contributes more fairly to the K-means distance calculation.

# Step 12: Apply the Elbow Method

In this step, the Elbow Method is used to help choose a suitable number of clusters.

The Elbow Method compares the K-means inertia values for different numbers of clusters. Inertia generally decreases as K increases, but the goal is to find a point where the improvement starts to slow down.

Unlike supervised learning, this assignment does not use train-test split, accuracy, precision, recall, F1-score, or a confusion matrix. Instead, the clustering result is evaluated using the Elbow Method and business interpretability.

In [ ]:
# Step 12: Apply the Elbow Method

# Create an empty list to store inertia values
inertia_values = []

# Create a range of K values to test
K_range = range(1, 11)

# Fit K-means for each K value and store inertia
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

# Create an elbow method table
elbow_table = pd.DataFrame({
    "K": list(K_range),
    "Inertia": inertia_values
})

# Save the elbow method values
elbow_table.to_csv(outputs_dir / "elbow_method_values.csv", index=False)

# Create the Elbow Method plot
plt.figure(figsize=(7, 4))
plt.plot(list(K_range), inertia_values, marker="o")
plt.title("Elbow Method for Choosing K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(K_range))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(outputs_dir / "elbow_method.png", dpi=300, bbox_inches="tight")
plt.show()

# Display the elbow method table
display(elbow_table)

## Short Interpretation

The Elbow Method shows how the inertia value changes as the number of clusters increases. The curve decreases quickly at first and then begins to flatten.

For this assignment, **K = 4** is selected because it provides a reasonable balance between technical separation and business interpretability. Four clusters also allow EventEdge to describe participants in practical business language.

# Step 13: Train the Final K-means Model and Add Cluster Labels

In this step, the final K-means model is trained using the selected number of clusters.

The cluster labels are then added back to the cleaned dataset. Since cluster numbers do not have business meaning by themselves, each cluster is also given a practical segment name.

In [ ]:
# Step 13: Train the Final K-means Model and Add Cluster Labels

# Set the final number of clusters
final_k = 4

# Create the final K-means model
kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)

# Fit the model and create cluster labels
cluster_labels = kmeans_final.fit_predict(X_scaled)

# Create a copy of the cleaned dataset and add cluster labels
df_clustered = df_clean.copy()
df_clustered["Cluster"] = cluster_labels

# Create practical business names for each cluster
segment_names = {
    0: "Active Short-Course Learners",
    1: "At-Risk Long-Course Low-Engagement Participants",
    2: "Premium Engaged Long-Course Participants",
    3: "Moderate-Engagement Short-Course Participants"
}

# Add segment names to the dataset
df_clustered["Segment_Name"] = df_clustered["Cluster"].map(segment_names)

# Save clustered results
# This file can be used for further business analysis or reporting
df_clustered.to_csv(outputs_dir / "eventedge_clustered_results.csv", index=False)

# Print confirmation message
print("Final K-means model trained successfully.")
print("Cluster labels and segment names have been added to the dataset.")

# Display segment counts
print("\nNumber of participants in each segment:")
display(df_clustered["Segment_Name"].value_counts())

# Display first five rows of clustered dataset
display(df_clustered.head())

## Short Interpretation

The final K-means model created four participant segments. These cluster labels have been added back to the dataset so that each participant can be connected to a segment.

The segment names help translate technical cluster numbers into practical business language.

# Step 14: Analyze and Interpret the Clusters

In this step, each cluster is profiled using average values of the selected numerical features.

Cluster profiling helps answer the main business questions:

- How many participant segments were found?
- What are the main characteristics of each segment?
- How are the segments different from each other?
- Which segment may be most valuable?
- Which segment may need more marketing or support attention?

In [ ]:
# Step 14: Analyze and Interpret the Clusters

# Calculate average values of selected features by cluster and segment name
cluster_profile = df_clustered.groupby(["Cluster", "Segment_Name"])[clustering_features].mean().round(2)

# Calculate number of participants in each cluster
cluster_sizes = df_clustered.groupby(["Cluster", "Segment_Name"]).size().rename("Number_of_Participants")

# Combine cluster sizes and average profiles
cluster_summary = pd.concat([cluster_sizes, cluster_profile], axis=1).reset_index()

# Add percentage of total participants
cluster_summary["Percent_of_Total"] = (
    cluster_summary["Number_of_Participants"] / len(df_clustered) * 100
).round(1)

# Add High_Dropout_Risk distribution only for interpretation, not training
if reference_column in df_clustered.columns:
    dropout_counts = pd.crosstab(df_clustered["Segment_Name"], df_clustered[reference_column])
    dropout_pct = pd.crosstab(
        df_clustered["Segment_Name"],
        df_clustered[reference_column],
        normalize="index"
    ).mul(100).round(1)

    if "Yes" in dropout_counts.columns:
        cluster_summary["Observed_Dropout_Risk_Yes_Count"] = (
            cluster_summary["Segment_Name"].map(dropout_counts["Yes"].to_dict()).fillna(0).astype(int)
        )
        cluster_summary["Observed_Dropout_Risk_Yes_Percent"] = (
            cluster_summary["Segment_Name"].map(dropout_pct["Yes"].to_dict()).fillna(0).round(1)
        )

# Save cluster summary
cluster_summary.to_csv(outputs_dir / "cluster_summary.csv", index=False)

# Display cluster summary
display(cluster_summary)

In [ ]:
# Create and display a dropout-risk distribution table by segment for business interpretation
if reference_column in df_clustered.columns:
    dropout_counts = pd.crosstab(df_clustered["Segment_Name"], df_clustered[reference_column])
    dropout_pct = pd.crosstab(
        df_clustered["Segment_Name"],
        df_clustered[reference_column],
        normalize="index"
    ).mul(100).round(1)

    dropout_distribution = pd.concat(
        [dropout_counts.add_suffix("_Count"), dropout_pct.add_suffix("_Percent")],
        axis=1
    ).fillna(0)

    dropout_distribution.to_csv(outputs_dir / "cluster_dropout_distribution.csv")

    print("High_Dropout_Risk Distribution by Segment")
    print("-" * 50)
    display(dropout_distribution)
else:
    print("No High_Dropout_Risk column is available for reference interpretation.")

## Cluster Interpretation

**Cluster 0: Active Short-Course Learners**  
This segment has shorter courses, lower fees, strong first-week attendance, high LMS logins, strong reminder-email engagement, and high satisfaction. These participants appear engaged and relatively low risk.

**Cluster 1: At-Risk Long-Course Low-Engagement Participants**  
This segment has longer and higher-fee courses, but low first-week attendance, low LMS logins, low reminder-email engagement, and the lowest satisfaction score. This segment needs the most marketing and support attention.

**Cluster 2: Premium Engaged Long-Course Participants**  
This segment has longer and higher-fee courses with good engagement and high satisfaction. This may be the most valuable segment because the participants are connected to higher-fee training and show positive engagement.

**Cluster 3: Moderate-Engagement Short-Course Participants**  
This segment has shorter and lower-fee courses with moderate engagement and medium satisfaction. This group may benefit from light reminders, onboarding support, and engagement nudges.

# Step 15: Visualize the Clusters

This step creates visualizations to support the cluster analysis and business interpretation.

The assignment requires at least two visualizations. This notebook includes multiple charts, including the Elbow Method plot, cluster size chart, engagement comparison chart, scatter plot, and PCA cluster visualization.

In [ ]:
# Chart 1: Number of Participants in Each Segment

# Set segment order based on cluster number
segment_order = [segment_names[i] for i in sorted(segment_names)]

# Count participants in each segment
cluster_size_plot = df_clustered["Segment_Name"].value_counts().reindex(segment_order)

# Create the chart
plt.figure(figsize=(10, 4))
ax = cluster_size_plot.plot(kind="bar")

# Add number labels on top of bars
for container in ax.containers:
    ax.bar_label(container)

# Add title and labels
plt.title("Number of Participants in Each Segment")
plt.xlabel("Segment")
plt.ylabel("Number of Participants")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(outputs_dir / "cluster_sizes.png", dpi=300, bbox_inches="tight")
plt.show()

## Chart Interpretation

The cluster size chart shows how many participants belong to each segment. This helps EventEdge understand whether each group is large enough to support practical marketing or participant-support actions.

In [ ]:
# Chart 2: Engagement Averages by Segment

# Select engagement-related features for comparison
engagement_features = [
    "Sessions_Attended_First_Week",
    "LMS_Logins_First_Week",
    "Reminder_Emails_Opened",
    "Satisfaction_Score_Week1"
]

# Calculate average engagement values by segment
engagement_averages = df_clustered.groupby("Segment_Name")[engagement_features].mean().reindex(segment_order).round(2)

# Create a grouped bar chart
ax = engagement_averages.plot(kind="bar", figsize=(11, 5))

# Add chart title and labels
plt.title("Average Engagement Measures by Segment")
plt.xlabel("Segment")
plt.ylabel("Average Value")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Engagement Measure")
plt.tight_layout()
plt.savefig(outputs_dir / "cluster_engagement_averages.png", dpi=300, bbox_inches="tight")
plt.show()

# Display the engagement averages table
display(engagement_averages)

## Chart Interpretation

The engagement averages chart compares attendance, LMS activity, email engagement, and first-week satisfaction across segments. It shows that some groups are highly engaged, while others show weaker early engagement and may need more support.

In [ ]:
# Chart 3: Scatter Plot Using Two Selected Features

# Create scatter plot using two business-relevant numerical features
plt.figure(figsize=(8, 5))

for cluster_id in sorted(df_clustered["Cluster"].unique()):
    cluster_data = df_clustered[df_clustered["Cluster"] == cluster_id]
    plt.scatter(
        cluster_data["LMS_Logins_First_Week"],
        cluster_data["Satisfaction_Score_Week1"],
        label=segment_names[cluster_id],
        alpha=0.7
    )

# Add chart title and labels
plt.title("Clusters by LMS Logins and Week 1 Satisfaction")
plt.xlabel("LMS Logins in First Week")
plt.ylabel("Satisfaction Score in Week 1")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(outputs_dir / "cluster_scatter_lms_satisfaction.png", dpi=300, bbox_inches="tight")
plt.show()

## Chart Interpretation

The scatter plot shows how participants are separated using LMS logins and first-week satisfaction. These two variables are useful because they reflect early engagement and participant experience.

In [ ]:
# Chart 4: PCA Visualization of Clusters

# Create a PCA object with two components
pca = PCA(n_components=2, random_state=42)

# Transform the scaled data into two principal components
pca_components = pca.fit_transform(X_scaled)

# Create a DataFrame for PCA visualization
pca_df = pd.DataFrame(pca_components, columns=["PC1", "PC2"])
pca_df["Cluster"] = df_clustered["Cluster"]
pca_df["Segment_Name"] = df_clustered["Segment_Name"]

# Create PCA scatter plot
plt.figure(figsize=(8, 5))

for cluster_id in sorted(pca_df["Cluster"].unique()):
    cluster_data = pca_df[pca_df["Cluster"] == cluster_id]
    plt.scatter(
        cluster_data["PC1"],
        cluster_data["PC2"],
        label=segment_names[cluster_id],
        alpha=0.7
    )

# Add chart title and labels
plt.title("PCA Visualization of K-means Clusters")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(outputs_dir / "pca_cluster_visualization.png", dpi=300, bbox_inches="tight")
plt.show()

# Display explained variance ratio
print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

## Chart Interpretation

The PCA chart reduces the selected clustering features into two dimensions so the clusters can be visualized more easily. This does not replace the full K-means model, but it provides a helpful visual summary of how the participant groups separate in a lower-dimensional space.

# Step 16: Business Interpretation and Recommendations

The purpose of clustering is not only to create groups, but also to explain how those groups can support better business decisions.

EventEdge can use the participant segments to improve marketing communication, onboarding, course support, and client relationship management.

In [ ]:
# Step 16: Business Interpretation and Recommendations

# Create a business recommendation table
business_recommendations = pd.DataFrame({
    "Segment": [
        "Active Short-Course Learners",
        "At-Risk Long-Course Low-Engagement Participants",
        "Premium Engaged Long-Course Participants",
        "Moderate-Engagement Short-Course Participants"
    ],
    "Business Meaning": [
        "Participants are active, satisfied, and responsive during the first week.",
        "Participants are in longer and higher-fee courses but show weak early engagement and low satisfaction.",
        "Participants are connected to higher-fee long courses and show strong engagement and satisfaction.",
        "Participants are in shorter courses with moderate engagement and may need light nudges."
    ],
    "Recommended Marketing or Support Strategy": [
        "Use as success cases, request feedback, and encourage referrals or advanced course enrollment.",
        "Prioritize early coaching, manager follow-up, onboarding support, and reminder communication.",
        "Protect this high-value group with premium support, progress updates, and advanced learning pathways.",
        "Send friendly reminders, simple LMS tips, progress nudges, and short motivational messages."
    ]
})

# Save business recommendations
business_recommendations.to_csv(outputs_dir / "business_recommendations_by_segment.csv", index=False)

# Display business recommendations
display(business_recommendations)

## Short Interpretation

The most important pattern is that participant groups differ strongly by early engagement and satisfaction. Participants with high LMS activity, good attendance, and high satisfaction appear more engaged, while participants with low activity and low satisfaction may need earlier support.

EventEdge should use the clusters as a decision-support tool. The segments can help the business prioritize coaching, reminders, onboarding support, and client communication without treating all participants the same.

# Step 17: Limitations and Responsible AI Reflection

K-means clustering is useful for discovering patterns, but it does not produce perfect customer or participant segments.

This section explains limitations of the dataset and model, possible bias concerns, and why human judgment should still be used when making marketing and participant-support decisions.

## Limitations and Responsible AI Reflection

One limitation is that the dataset is synthetic and simplified. It may not capture all real-world reasons for dropout or engagement, such as personal circumstances, manager communication quality, workplace pressure, illness, time-zone issues, or organizational culture.

K-means does not always produce perfect customer or participant segments because it groups records based on distance. The result can change depending on the selected features, scaling method, number of clusters, and the shape of the data. K-means also works best when clusters are roughly compact and separated, which may not always match real business behaviour.

A possible unfair decision could happen if EventEdge uses clusters without human review. For example, a low-engagement participant could be treated as unmotivated when the real issue may be workload, manager support, scheduling difficulty, or technical access problems. This could lead to unfair or insensitive communication.

Human judgment is still important because managers and training advisors can review context before taking action. The clusters should be used to guide helpful support, not to blame participants or automatically reduce service quality. Responsible use means providing respectful reminders, coaching, flexible support, and fair review of segment-based decisions.

# Step 18: Assignment and Week 5 Alignment Checklist

This final section checks that the notebook follows the Assignment 5 requirements and the Week 5 unsupervised learning lecture flow.

In [ ]:
# Step 18: Assignment and Week 5 Alignment Checklist

# Create assignment alignment checklist
alignment_checklist = pd.DataFrame({
    "Requirement": [
        "Understand the business problem",
        "Load and inspect the dataset",
        "Check missing values and duplicate rows",
        "Clean the dataset",
        "Select useful numerical features",
        "Scale selected features",
        "Use the Elbow Method",
        "Train final K-means model",
        "Add cluster labels back to dataset",
        "Analyze and interpret clusters",
        "Create visualizations",
        "Provide business recommendations",
        "Discuss limitations and responsible AI"
    ],
    "Where Addressed": [
        "Step 1",
        "Steps 3 and 4",
        "Step 5",
        "Step 6",
        "Step 10",
        "Step 11",
        "Step 12",
        "Step 13",
        "Step 13",
        "Step 14",
        "Step 15",
        "Step 16",
        "Step 17"
    ]
})

# Save assignment alignment checklist
alignment_checklist.to_csv(outputs_dir / "assignment_alignment_checklist.csv", index=False)

# Create Week 5 alignment checklist
week5_alignment_checklist = pd.DataFrame({
    "Week 5 Concept": [
        "Unsupervised learning has no target variable for training",
        "Customer/user segmentation is a business use case",
        "K-means groups similar records into clusters",
        "Selected numerical features should be scaled",
        "The Elbow Method helps choose K",
        "Clusters should be interpreted in business language",
        "Visualizations help explain segments",
        "Responsible AI and human judgment are still required"
    ],
    "How This Notebook Applies It": [
        "High_Dropout_Risk is excluded from K-means training",
        "EventEdge participants are grouped into training-support segments",
        "KMeans is used to create four participant segments",
        "StandardScaler is used before K-means",
        "Inertia is compared for K = 1 to 10",
        "Cluster numbers are converted into segment names",
        "Elbow, bar, scatter, and PCA charts are created",
        "Limitations, fairness, and human review are discussed"
    ]
})

# Save Week 5 alignment checklist
week5_alignment_checklist.to_csv(outputs_dir / "week5_lecture_alignment_checklist.csv", index=False)

# Display both checklists
print("Assignment Alignment Checklist")
print("-" * 50)
display(alignment_checklist)

print("Week 5 Lecture Alignment Checklist")
print("-" * 50)
display(week5_alignment_checklist)

# Conclusion

This notebook applied K-means clustering to discover participant segments for EventEdge Corporate Training. The workflow followed the Assignment 5 requirements and the Week 5 unsupervised learning approach.

The final model identified four participant segments. The results suggest that EventEdge can use early engagement and satisfaction patterns to improve participant support, marketing communication, and client training outcomes. The analysis should be used as a decision-support tool, and human judgment should guide final business actions.